In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction==2026.11.0 --extra-index-url https://download.easydiffraction.org/pypi/simple/


# Single-crystal fit: taurine, SENJU

Fit the amino-sulfonic acid taurine to six time-of-flight neutron single-crystal datasets from
SENJU at J-PARC together, each reflection measured at its own wavelength.

The project is the catalogue case `sc-neut-tof_taurine-senju`, the one the command line and the
app's Examples open. Banks 2 to 6 share bank 1's three extinction terms, from the start.

<span style="color:#398648;background:#39864821;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Fitting</span> <span style="color:#96601d;background:#96601d21;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Joint</span> <span style="color:#424242;background:#42424221;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">sc</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">neut</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">tof</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Bragg</span> <span style="color:#00796b;background:#00796b21;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Crystal structure</span>

In [2]:
import easydiffraction as edi

## Load the project

In [3]:
project_path = edi.download_data('sc-neut-tof_taurine-senju', destination='data')
project = edi.Project.load(project_path)
structure = project.structures['taurine']
senju_1 = project.experiments['senju_1']
senju_2 = project.experiments['senju_2']
senju_3 = project.experiments['senju_3']
senju_4 = project.experiments['senju_4']
senju_5 = project.experiments['senju_5']
senju_6 = project.experiments['senju_6']
project.display.constraints()

Data 'sc-neut-tof_taurine-senju' is at data/sc-neut-tof_taurine-senju-b5deb357e7e8/sc-neut-tof_taurine-senju


alias,parameter
x11_1,senju_1.extinction.coefficient_11
x22_1,senju_1.extinction.coefficient_22
x33_1,senju_1.extinction.coefficient_33
x11_2,senju_2.extinction.coefficient_11
x22_2,senju_2.extinction.coefficient_22
x33_2,senju_2.extinction.coefficient_33
x11_3,senju_3.extinction.coefficient_11
x22_3,senju_3.extinction.coefficient_22
x33_3,senju_3.extinction.coefficient_33
x11_4,senju_4.extinction.coefficient_11


The project marks what its fit frees. The fit below frees it in stages, so start with all of it
fixed.

In [4]:
for parameter in project.free_parameters:
    parameter.free = False

In [5]:
project.display.pattern('senju_1')

View(html=<119,465 characters>, height=480)

## Perform the analysis

The fit follows diffraction-lib's change of model for this case: isotropic displacements first,
then anisotropic tensors. diffraction-lib does not fit the positions, which this case does, so
they get a stage of their own.

### Scales, every site isotropic

Each bank's scale, with every site's displacement isotropic for now.

In [6]:
for site in structure.atom_sites:
    site.adp_type = 'Biso'
for experiment in project.experiments:
    experiment.linked_structures['taurine'].scale.free = True

In [7]:
project.display.parameters.free()

#,datablock,category,entry,parameter,value,s.u.,min,max,units
1,senju_1,linked_structure,taurine,scale,2.762,,-inf,inf,
2,senju_2,linked_structure,taurine,scale,3.829,,-inf,inf,
3,senju_3,linked_structure,taurine,scale,3.211,,-inf,inf,
4,senju_4,linked_structure,taurine,scale,2.809,,-inf,inf,
5,senju_5,linked_structure,taurine,scale,2.739,,-inf,inf,
6,senju_6,linked_structure,taurine,scale,2.936,,-inf,inf,


In [8]:
project.analysis.fit()
project.display.fit.results()

edi fit: joint project sc-neut-tof_taurine-senju, 6 banks (senju_1, senju_2, senju_3, senju_4, senju_5, senju_6), 947 pts fitted / 1540 loaded, 6 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                       179.47      5665.97
      1      0.003      51.73       470.74   91.7% ↓
      2      0.005      51.73       470.73   
  done · converged · 2 iters · reduced χ² 470.73 · wR(F²) 51.73 % · 0.01 s


View(html=<95,631 characters>, height=376)

In [9]:
project.display.pattern('senju_1')

View(html=<119,154 characters>, height=480)

### Isotropic displacements

Every site's isotropic displacement.

In [10]:
for site in structure.atom_sites:
    site.adp_iso.free = True

In [11]:
project.analysis.fit()
project.display.fit.results()

edi fit: joint project sc-neut-tof_taurine-senju, 6 banks (senju_1, senju_2, senju_3, senju_4, senju_5, senju_6), 947 pts fitted / 1540 loaded, 20 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                        51.73       477.84
      1      0.005      31.17       173.48   63.7% ↓
      2      0.009      27.18       131.89   24.0% ↓
      3      0.014      26.94       129.62   1.7% ↓
      4      0.018      26.94       129.57   
      5      0.023      26.94       129.56   
      6      0.028      26.94       129.56   
  done · converged · 6 iters · reduced χ² 129.56 · wR(F²) 26.94 % · 0.03 s


View(html=<110,538 characters>, height=376)

In [12]:
project.display.pattern('senju_1')

View(html=<119,600 characters>, height=480)

### Extinction

Bank 1's three extinction terms, which the other banks share.

In [13]:
senju_1.extinction.coefficient_11.free = True
senju_1.extinction.coefficient_22.free = True
senju_1.extinction.coefficient_33.free = True

In [14]:
project.analysis.fit()
project.display.fit.results()

edi fit: joint project sc-neut-tof_taurine-senju, 6 banks (senju_1, senju_2, senju_3, senju_4, senju_5, senju_6), 947 pts fitted / 1540 loaded, 23 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                        26.94       129.98
      1      0.004      22.14        87.84   32.4% ↓
      2      0.008      20.47        75.09   14.5% ↓
      3      0.011      20.27        73.57   2.0% ↓
      4      0.015      20.26        73.55   
      5      0.019      20.26        73.55   
  done · converged · 5 iters · reduced χ² 73.55 · wR(F²) 20.26 % · 0.02 s


View(html=<114,231 characters>, height=376)

In [15]:
project.display.pattern('senju_1')

View(html=<119,769 characters>, height=480)

### Positions

All fourteen sites' coordinates.

In [16]:
for site in structure.atom_sites:
    site.fract_x.free = True
    site.fract_y.free = True
    site.fract_z.free = True

In [17]:
project.analysis.fit()
project.display.fit.results()

edi fit: joint project sc-neut-tof_taurine-senju, 6 banks (senju_1, senju_2, senju_3, senju_4, senju_5, senju_6), 947 pts fitted / 1540 loaded, 65 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                        20.26        77.05
      1      0.008      17.17        55.36   28.2% ↓
      2      0.016      17.04        54.51   1.5% ↓
      3      0.022      17.04        54.49   
      4      0.028      17.04        54.49   
  done · converged · 4 iters · reduced χ² 54.49 · wR(F²) 17.04 % · 0.04 s


View(html=<158,829 characters>, height=376)

In [18]:
project.display.pattern('senju_1')

View(html=<119,870 characters>, height=480)

### Anisotropic displacements

Back to anisotropic displacements (edi converts the values), with every tensor component free.

In [19]:
for site in structure.atom_sites:
    site.adp_type = 'beta'
for tensor in structure.atom_site_aniso:
    tensor.adp_11.free = True
    tensor.adp_22.free = True
    tensor.adp_33.free = True
    tensor.adp_12.free = True
    tensor.adp_13.free = True
    tensor.adp_23.free = True

/tmp/ipykernel_4224/4011545669.py:4: UserWarning: crysta.domain.dependent_free_ignored: parameter 'taurine.atom_site.S1.adp_iso' is constrained by 'the equivalent isotropic value of its beta tensor'; free = True is ignored and it stays dependent (cannot be fitted: set by a constraint)
  tensor.adp_11.free = True
/tmp/ipykernel_4224/4011545669.py:4: UserWarning: crysta.domain.dependent_free_ignored: parameter 'taurine.atom_site.O1.adp_iso' is constrained by 'the equivalent isotropic value of its beta tensor'; free = True is ignored and it stays dependent (cannot be fitted: set by a constraint)
  tensor.adp_11.free = True
/tmp/ipykernel_4224/4011545669.py:4: UserWarning: crysta.domain.dependent_free_ignored: parameter 'taurine.atom_site.O2.adp_iso' is constrained by 'the equivalent isotropic value of its beta tensor'; free = True is ignored and it stays dependent (cannot be fitted: set by a constraint)
  tensor.adp_11.free = True
/tmp/ipykernel_4224/4011545669.py:4: UserWarning: crysta.d

In [20]:
project.analysis.fit()
project.display.fit.results()

edi fit: joint project sc-neut-tof_taurine-senju, 6 banks (senju_1, senju_2, senju_3, senju_4, senju_5, senju_6), 947 pts fitted / 1540 loaded, 135 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                        17.04        59.19
      1      0.022      13.02        34.56   41.6% ↓
      2      0.037      12.20        30.32   12.3% ↓
      3      0.058      12.19        30.27   0.2% ↓
      4      0.073      12.19        30.27   
  done · converged · 4 iters · reduced χ² 30.27 · wR(F²) 12.19 % · 0.13 s


View(html=<234,097 characters>, height=376)

In [21]:
project.display.pattern('senju_1')

View(html=<121,042 characters>, height=480)

In [22]:
project.display.fit.correlations()

,1,2,3,4
1 taurine.atom_site.N1.fract_x,,,,
2 taurine.atom_site.H3.fract_x,-0.54,,,
3 senju_1.linked_structure.taurine.scale,0.04,-0.13,,
4 senju_1.extinction.coefficient_11,-0.10,0.08,0.50,


In [23]:
project.display.structure('taurine')

View(html=<274,945 characters>, height=660)

## Save the project

In [24]:
project.save_as('sc-neut-tof_taurine-senju')
project.report.save_html('sc-neut-tof_taurine-senju.html')

Saving project to 'sc-neut-tof_taurine-senju'
Saving report to 'sc-neut-tof_taurine-senju.html'
